# Conatus Phronesis — merge + quantização GGUF pra rodar no Ollama

Funde o adapter LoRA com o base `ibm-granite/granite-4.1-8b`, converte pra GGUF e quantiza.
Runtime: GPU (T4/L4) acelera o merge; a conversão/quantização em si roda em CPU.

**Este notebook é para o DEPLOY, não para avaliar o modelo.** Para medir se a troca de
base ajudou o raciocínio, use as células de avaliação do `train_colab.ipynb`, que rodam o
modelo direto em `transformers` sobre o adapter — sem passar por GGUF. Assim o resultado
mede o modelo, e não o modelo mais o efeito da quantização. Vir para cá antes disso
mistura duas variáveis e você não saberá a qual atribuir uma eventual piora.

**Escolha da quantização.** Com o Granite 4.1 **8B** (dense) os números mudaram de figura
em relação ao 14B — agora sobra espaço:

| Quant | Tamanho (8B) | Onde cabe |
|---|---|---|
| Q8_0 | ~8,5 GB | L4/A100 |
| Q6_K | ~6,6 GB | L4/A100; GPU de 8 GB no limite |
| Q5_K_M | ~5,7 GB | L4/A100 e GPU de 8 GB |
| **Q4_K_M** | **~4,9 GB** | **qualquer uma das acima — padrão** |

O default segue `Q4_K_M` pelo equilíbrio tamanho/qualidade e pela comparabilidade com as
rodadas anteriores. Mas repare: no 8B até o **Q6_K cabe no L4 com folga**, então se a
avaliação apontar perda de rigor matemático depois da quantização, subir para Q5_K_M/Q6_K
é barato — o que não era verdade no 14B. As linhas Q3/IQ3 saíram da tabela: não há mais
motivo para descer até a família que o próprio llama.cpp classifica como "low quality".

**Arquitetura**: o Granite 4.1 8B é um transformer *dense* (`GraniteForCausalLM`), não um
híbrido Mamba como os `granite-4.0-h-*`. O llama.cpp converte direto (`conversion/granite.py`)
e a IBM publica GGUFs oficiais do mesmo modelo — nada de especial no caminho abaixo.

**Pré-requisito**: secret `HF_TOKEN` no Colab (ícone de chave) com acesso ao repo
privado do adapter.


In [ ]:
# 1) Login HF
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

# Alvo atual do branch (migrado do Qwen3-14B em 2026-07-28). Para refazer o A/B com o
# modelo anterior, troque para "Qwen/Qwen3-14B" e use o secret PHRONESIS_14B_ADAPTER_REPO.
# O adapter TEM que ter sido treinado sobre o mesmo base: LoRA nao transfere entre familias.
BASE_MODEL = "ibm-granite/granite-4.1-8b"
ADAPTER_REPO = userdata.get("PHRONESIS_GRANITE8B_ADAPTER_REPO")
if not ADAPTER_REPO:
    raise ValueError("Configure o secret PHRONESIS_GRANITE8B_ADAPTER_REPO com o repo do adapter")
MERGED_DIR = "/content/merged"
GGUF_F16 = "/content/model-f16.gguf"

# Q4_K_M (~4,9GB no 8B): padrao de equilibrio tamanho/qualidade. No 8B ha folga de sobra —
# subir pra Q5_K_M (~5,7GB) ou Q6_K (~6,6GB) e barato e ainda cabe ate numa GPU de 8GB.
# Ver a tabela na celula acima.
QUANT_TYPE = "Q4_K_M"
GGUF_QUANT = f"/content/model-{QUANT_TYPE}.gguf"


In [ ]:
# 2) Deps pro merge
%pip install -q -U transformers accelerate peft safetensors


In [ ]:
# 3) Merge: baixa base + adapter, funde os pesos, salva em formato HF (safetensors)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Carregando base...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="cpu")

print("Carregando adapter (repo privado)...")
model = PeftModel.from_pretrained(base, ADAPTER_REPO)

print("Fundindo LoRA nos pesos do base...")
model = model.merge_and_unload()

print(f"Salvando modelo fundido em {MERGED_DIR}...")
model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("Merge concluido.")


In [ ]:
# 4) llama.cpp: clona e instala os requirements do script de conversao
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
%pip install -q -r /content/llama.cpp/requirements.txt


In [ ]:
# 5) Converte o modelo fundido (HF/safetensors) pra GGUF f16
!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_F16} --outtype f16


In [ ]:
# 6) Compila o llama.cpp (cmake) pra ter o binario de quantizacao
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DCMAKE_BUILD_TYPE=Release
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4


In [ ]:
# 7) Quantiza o f16 pro tipo escolhido
!/content/llama.cpp/build/bin/llama-quantize {GGUF_F16} {GGUF_QUANT} {QUANT_TYPE}

import os
size_gb = os.path.getsize(GGUF_QUANT) / (1024**3)
print(f"\nGGUF quantizado: {GGUF_QUANT} ({size_gb:.2f} GB)")
print("Confira com `ollama ps` depois de carregar: se size_vram == size, coube 100% na GPU.")


In [ ]:
# 8) Baixa o arquivo final pro seu computador
from google.colab import files
files.download(GGUF_QUANT)


## 9) (opcional) Persistir o GGUF no HF Hub em vez de baixar

Util se a conexao cair no meio do download, ou pra reusar depois sem regerar.


In [ ]:
# Opcional: sobe o GGUF pro mesmo repo privado (evita perder se o download falhar)
from huggingface_hub import HfApi
HfApi().upload_file(
    path_or_fileobj=GGUF_QUANT,
    path_in_repo=f"gguf/model-{QUANT_TYPE}.gguf",
    repo_id=ADAPTER_REPO,
    repo_type="model",
)
print("GGUF tambem disponivel em:", f"https://huggingface.co/{ADAPTER_REPO}/blob/main/gguf/model-{QUANT_TYPE}.gguf")


## 10) Rodar no Ollama

Com o arquivo `.gguf` em mãos, crie um `Modelfile` ao lado dele:

```
FROM ./model-Q4_K_M.gguf

PARAMETER temperature 0.6
PARAMETER top_p 0.95
PARAMETER top_k 20
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 4096
PARAMETER stop "<|end_of_text|>"
```

Depois:

```bash
ollama create phronesis-granite-8b -f Modelfile
ollama run phronesis-granite-8b
```

Confira que coube inteiro na GPU com `ollama ps`: se `size_vram` bater com `size`, está
100% na GPU. Se houver divisão CPU/GPU, o modelo transbordou e a latência vai sofrer —
quantize mais agressivamente ou sirva num host com mais VRAM.

**Sobre a decodificação**: `temperature 0.6` + `top_p 0.95` + `top_k 20` são herdados da
rodada Qwen3 — o `generation_config.json` do Granite 4.1 não publica parâmetros de
sampling. Foram mantidos na troca de base para não mexer em duas variáveis de uma vez, e
são os mesmos que `src/eval_harness.py` usa, então a demo bate com o eval. O `Modelfile` antigo
trazia `temperature 0` (greedy) com uma nota afirmando que greedy era melhor — essa
conclusão é **anterior** ao experimento registrado em
`data/eval/thinking_8b_eval_notes.md`, que mediu greedy puro entrando em loop de
repetição em 4 de 16 itens. O loop some com sampling. Não volte pra `temperature 0` sem
reler aquelas notas. O `phronesis-4b` que está em produção hoje ainda usa `temperature 0`.

O GGUF já carrega o chat template do Granite embutido (inclusive o formato `<tool_call>`),
então o Ollama renderiza tools automaticamente via `/api/chat` com o parâmetro `tools`.

O `<think>`, por outro lado, **não vem do template** — o Granite não tem canal de reasoning.
Ele sai do modelo treinado, como texto normal do turno do assistant, e `<think>`/`</think>`
são tokens do vocabulário. Na prática isso significa que o cliente que consumir a API
precisa separar o bloco por conta própria (mesmo que `strip_think_blocks()` em
`src/common.py` faz), em vez de esperar um campo `thinking` separado na resposta.
